# NV-NMR Template Parameter, Table, And Waveform Editing

This notebook is the first plain Python application notebook for the clean NV-NMR TNMR templates:

- `20260620_NVNMR_template.tps`: the editable sequence file
- `20260620_NVNMR_template.tnt`: the TNMR document that embeds the same sequence payload

The goal is simple: load the files, extract the sequence information, extract parameters, extract modulation tables, visualize them, edit selected values, apply a generated waveform to a modulation table, and write TNMR-readable output files.

Keep this notebook conservative. The `.tps` path is the source of truth for sequence edits. The `.tnt` write path replaces the embedded sequence payload in this clean paired template and should be accepted by opening the output in TNMR before using it operationally.


## 1. Setup

Find the repo root, import the Redstone helpers, and choose a generated-output folder. The helper classes are repo-local; no TNMR COM connection is needed for the `.tps` workflow.


In [ ]:
import hashlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        redstone_dir = candidate / "hardware_capture" / "rf" / "redstone"
        if (redstone_dir / "sequence.py").exists() and (redstone_dir / "waveform.py").exists():
            return candidate
    raise FileNotFoundError("Could not find the Hardware Capture repo root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REDSTONE_DIR = REPO_ROOT / "hardware_capture" / "rf" / "redstone"
EXAMPLES_DIR = REDSTONE_DIR / "examples"
OUTPUT_DIR = EXAMPLES_DIR / "generated" / "nvnmr_template"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from hardware_capture.rf.redstone import Sequence, Sine

print("Repo root:", REPO_ROOT)
print("Output dir:", OUTPUT_DIR)


## 2. Select The Clean Templates

The notebook uses the clean template pair in `hardware_capture/rf/redstone`. Output files are written separately under `examples/generated/nvnmr_template`.


In [ ]:
TEMPLATE_TPS_PATH = REDSTONE_DIR / "20260620_NVNMR_template.tps"
TEMPLATE_TNT_PATH = REDSTONE_DIR / "20260620_NVNMR_template.tnt"

OUTPUT_TPS_PATH = OUTPUT_DIR / "20260620_NVNMR_template_waveform_edit.tps"
OUTPUT_TNT_PATH = OUTPUT_DIR / "20260620_NVNMR_template_waveform_edit.tnt"
OUTPUT_JSON_PATH = OUTPUT_DIR / "20260620_NVNMR_template_waveform_edit.tps.json"

assert TEMPLATE_TPS_PATH.exists(), TEMPLATE_TPS_PATH
assert TEMPLATE_TNT_PATH.exists(), TEMPLATE_TNT_PATH

TEMPLATE_TPS_PATH, TEMPLATE_TNT_PATH


## 3. Load Files And Check The TNT Embedded Sequence

The clean `.tnt` file embeds the same sequence bytes as the `.tps` file, but TNMR stores the embedded sequence signature in a compact header form. These helper functions only handle this clean paired-template case.


In [ ]:
TNT_EMBEDDED_PSEQ_SIGNATURE = b"PSEQ" + (1).to_bytes(4, "little") + b"1.18 BIN"
TPS_PSEQ_SIGNATURE = b"PSEQ1.001.18 BIN"


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def tps_bytes_for_tnt(tps_bytes: bytes) -> bytes:
    if not tps_bytes.startswith(TPS_PSEQ_SIGNATURE):
        raise ValueError("TPS bytes do not start with the expected template PSEQ signature.")
    return TNT_EMBEDDED_PSEQ_SIGNATURE + tps_bytes[16:]


def tps_bytes_from_tnt(tnt_bytes: bytes, tps_size: int) -> tuple[bytes, int]:
    start = tnt_bytes.find(TNT_EMBEDDED_PSEQ_SIGNATURE)
    if start < 0:
        raise ValueError("Could not find the embedded PSEQ payload in the TNT file.")
    embedded = tnt_bytes[start : start + tps_size]
    return TPS_PSEQ_SIGNATURE + embedded[16:], start


tps_bytes = TEMPLATE_TPS_PATH.read_bytes()
tnt_bytes = TEMPLATE_TNT_PATH.read_bytes()
embedded_tps_bytes, embedded_tps_offset = tps_bytes_from_tnt(tnt_bytes, len(tps_bytes))

assert embedded_tps_bytes == tps_bytes

file_summary = pd.DataFrame(
    [
        {
            "file": TEMPLATE_TPS_PATH.name,
            "bytes": len(tps_bytes),
            "sha256": sha256_bytes(tps_bytes),
        },
        {
            "file": TEMPLATE_TNT_PATH.name,
            "bytes": len(tnt_bytes),
            "sha256": sha256_bytes(tnt_bytes),
        },
    ]
)

display(file_summary)
print("Embedded TPS offset inside TNT:", embedded_tps_offset)


## 4. Extract Sequence Summary

`Sequence.load()` parses the TPS header, visible sequence grid, table references, named table definitions, and parameter definitions.


In [ ]:
sequence = Sequence.load(TEMPLATE_TPS_PATH)
summary = sequence.summary().to_dict()

summary_for_display = {
    key: value
    for key, value in summary.items()
    if key not in {"control_lines", "parameter_references"}
}

display(pd.Series(summary_for_display, name="value"))
print("Control lines:", len(summary["control_lines"]))
print("Parameter references:", ", ".join(summary["parameter_references"]))


## 5. Extract Parameters

The template has an extended TNMR parameter record. `value` is the main sequence value. `current_value` is TNMR's duplicate/current slot when present; parameter edits in `Sequence.set_parameter_values()` update both so the output stays internally consistent.


In [ ]:
parameters_df = pd.DataFrame(
    [
        {
            "name": parameter["name"],
            "value": parameter["value"],
            "current_value": parameter.get("current_value", ""),
            "minimum": parameter.get("minimum", ""),
            "maximum": parameter.get("maximum", ""),
            "type": parameter.get("parameter_type"),
            "read_only": parameter.get("read_only"),
        }
        for parameter in sequence.parameters
    ]
)

referenced = set(sequence.current_parameter_references)
defined = set(sequence.parameter_values())

parameters_df["referenced_by_rows"] = parameters_df["name"].isin(referenced)
display(parameters_df)

print("Referenced but not defined:", sorted(referenced - defined))
print("Defined but not referenced:", sorted(defined - referenced))


## 6. Extract Named Tables

The modulation tables are stored as named table definitions. The table reference grid below tells which visible sequence events point at each named table.


In [ ]:
def table_entries(table: dict) -> list[str]:
    text = str(table.get("entries", "")).replace("\r\n", "\n").strip()
    return [] if not text else text.splitlines()


tables_df = pd.DataFrame(
    [
        {
            "name": table["name"],
            "entry_count": len(table_entries(table)),
            "table_type": table.get("table_type_text"),
            "dimension": table.get("dimension"),
            "steps_per_360_cycle": table.get("steps_per_360_cycle"),
            "value_type": table.get("value_type"),
            "entries_preview": " ".join(table_entries(table)[:8]),
        }
        for table in sequence.tables
    ]
)

table_refs_df = pd.DataFrame(reference.to_dict() for reference in sequence.table_references())
if not table_refs_df.empty:
    table_refs_df = table_refs_df[table_refs_df["table_name"] != ""]

print("Named tables")
display(tables_df)
print("Visible event table references")
display(table_refs_df)


## 7. Preview The Sequence Grid

This compact view shows the first visible events. Cells with table references include the table name in brackets.


In [ ]:
def cell_text(cell) -> str:
    text = str(cell.data)
    if cell.table_references:
        text = f"{text} [{', '.join(cell.table_references)}]"
    return text


preview_rows = []
for row in sequence.table_view().rows:
    record = {"line": row.label}
    for cell in row.cells[:12]:
        record[f"event_{cell.event_index + 1}"] = cell_text(cell)
    preview_rows.append(record)

sequence_preview_df = pd.DataFrame(preview_rows).set_index("line")
display(sequence_preview_df)


## 8. Visualize Current Modulation Tables

Use this plot to inspect table values before changing them. Phase tables are expected to be numeric degrees or phase-step values, depending on the TNMR table metadata.


In [ ]:
def numeric_table_df(sequence: Sequence, table_name: str) -> pd.DataFrame:
    table = next(table for table in sequence.tables if table["name"] == table_name)
    rows = []
    for index, text in enumerate(table_entries(table)):
        try:
            value = float(text)
        except ValueError:
            value = None
        rows.append({"index": index, "value": value, "text": text})
    return pd.DataFrame(rows)


for table_name in ["ph_cycle", "phmod_p180", "phmod_pn180"]:
    df = numeric_table_df(sequence, table_name)
    print(f"{table_name}: {len(df)} entries")
    display(df.head(16))
    if not df.empty and df["value"].notna().any():
        ax = df.plot(x="index", y="value", marker="o", title=table_name, legend=False)
        ax.set_xlabel("table index")
        ax.set_ylabel("value")
        plt.show()


## 9. Define The Application Edit

For the first application case, generate a phase waveform, write it into the existing `phmod_pn180` F1 phase-modulation table, then explicitly assign that table to the target visible event. The table is referenced by several visible events, so `allow_shared=True` is intentional when editing its contents: every event using that table receives the same waveform.


In [ ]:
PARAMETER_UPDATES = {
    "N_4fcycle": 12,
    "t_pn180": "12u",
    "t_mod_offset": "220n",
}

TARGET_TABLE_NAME = "phmod_pn180"
TARGET_LINE = "F1_PhMod"
TARGET_EVENT_NUMBER = 5

waveform = Sine(
    points=32,
    phase_min=0.0,
    phase_max=180.0,
    amp_min=None,
    amp_max=None,
    phase_steps=1024,
)

waveform_df = pd.DataFrame(waveform.sample_records())
display(pd.Series(waveform.summary().to_dict(), name="value"))
display(waveform_df.head())

ax = waveform_df.plot(x="index", y="phase_deg", marker="o", legend=False)
ax.set_title("Generated phase waveform")
ax.set_xlabel("table index")
ax.set_ylabel("phase (deg)")
plt.show()


## 10. Apply The Edit In Python

Reload the clean template, apply parameter changes, replace the target modulation table entries, and inspect the before/after values before writing files.


In [ ]:
edited = Sequence.load(TEMPLATE_TPS_PATH)

before_parameters_df = pd.DataFrame(
    [{"name": name, "before": value} for name, value in edited.parameter_values().items()]
)
before_table_df = numeric_table_df(edited, TARGET_TABLE_NAME)

edited.set_parameter_values(PARAMETER_UPDATES)
edited.set_waveform_to_table(
    TARGET_TABLE_NAME,
    waveform,
    quantity="phase",
    allow_shared=True,
)
edited.assign_table_to_event(
    TARGET_LINE,
    TARGET_EVENT_NUMBER,
    TARGET_TABLE_NAME,
)

after_parameters_df = pd.DataFrame(
    [{"name": name, "after": value} for name, value in edited.parameter_values().items()]
)
after_table_df = numeric_table_df(edited, TARGET_TABLE_NAME)

parameter_diff_df = before_parameters_df.merge(after_parameters_df, on="name")
parameter_diff_df = parameter_diff_df[parameter_diff_df["before"] != parameter_diff_df["after"]]

display(parameter_diff_df)
display(after_table_df.head(40))


## 11. Visualize Before And After

The original `phmod_pn180` table is a single zero. The edited table is the generated waveform column.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
if not before_table_df.empty:
    ax.plot(before_table_df["index"], before_table_df["value"], "o-", label="before")
if not after_table_df.empty:
    ax.plot(after_table_df["index"], after_table_df["value"], "o-", label="after")
ax.set_title(f"{TARGET_TABLE_NAME} before/after")
ax.set_xlabel("table index")
ax.set_ylabel("phase (deg)")
ax.legend()
plt.show()


## 12. Write TPS And JSON, Then Reload

The reload check confirms that the edited `.tps` can be parsed again and that the intended parameter/table changes are present.


In [ ]:
edited.write_tps(OUTPUT_TPS_PATH)
edited.write_json(OUTPUT_JSON_PATH, include_raw_bytes=False)

reloaded = Sequence.load(OUTPUT_TPS_PATH)
reloaded_table = next(table for table in reloaded.tables if table["name"] == TARGET_TABLE_NAME)
reloaded_n_4fcycle = next(
    parameter for parameter in reloaded.parameters if parameter["name"] == "N_4fcycle"
)

assert reloaded.parameter_values()["t_pn180"] == "12u"
assert reloaded_n_4fcycle["value"] == "12"
assert reloaded_n_4fcycle.get("current_value") == "12"
assert len(table_entries(reloaded_table)) == waveform.summary().point_count

print("Wrote TPS:", OUTPUT_TPS_PATH)
print("Wrote JSON:", OUTPUT_JSON_PATH)
print("Reloaded table entries:", len(table_entries(reloaded_table)))


## 13. Write A TNT Copy With The Edited Embedded TPS

For this clean paired template, the original TNT contains the original TPS payload with a compact TNMR embedded signature. This cell replaces that payload with the edited TPS payload and preserves the TNT bytes before and after the embedded sequence.

This is intentionally transparent and narrow. Open the generated `.tnt` in TNMR before operational use. If TNMR rewrites or rejects a length-changing embedded sequence, use TNMR COM to open the original TNT, load the edited TPS, and `SaveAs` from TNMR itself.


In [ ]:
def write_tnt_with_edited_tps(
    template_tnt_path: Path,
    template_tps_path: Path,
    edited_tps_path: Path,
    output_tnt_path: Path,
) -> Path:
    template_tnt_bytes = template_tnt_path.read_bytes()
    template_tps_bytes = template_tps_path.read_bytes()
    edited_tps_bytes = edited_tps_path.read_bytes()

    original_embedded = tps_bytes_for_tnt(template_tps_bytes)
    edited_embedded = tps_bytes_for_tnt(edited_tps_bytes)

    start = template_tnt_bytes.find(original_embedded)
    if start < 0:
        raise ValueError("The template TNT does not contain the expected template TPS payload.")

    output_bytes = (
        template_tnt_bytes[:start]
        + edited_embedded
        + template_tnt_bytes[start + len(original_embedded) :]
    )
    output_tnt_path.write_bytes(output_bytes)
    return output_tnt_path


write_tnt_with_edited_tps(
    TEMPLATE_TNT_PATH,
    TEMPLATE_TPS_PATH,
    OUTPUT_TPS_PATH,
    OUTPUT_TNT_PATH,
)

edited_tnt_bytes = OUTPUT_TNT_PATH.read_bytes()
restored_tps_bytes, restored_tps_offset = tps_bytes_from_tnt(
    edited_tnt_bytes,
    len(OUTPUT_TPS_PATH.read_bytes()),
)
assert restored_tps_bytes == OUTPUT_TPS_PATH.read_bytes()

print("Wrote TNT:", OUTPUT_TNT_PATH)
print("Edited embedded TPS offset:", restored_tps_offset)
print("Edited TNT bytes:", len(edited_tnt_bytes))


## 14. Optional TNMR COM Save Path

Leave this disabled unless TNMR is installed and you want TNMR itself to perform the final TNT save. This is useful if engineering decides the production path should avoid direct TNT byte replacement.


In [ ]:
RUN_TNMR_COM_SAVE = True

if RUN_TNMR_COM_SAVE:
    from hardware_capture.rf.redstone import TNMRControl

    control = TNMRControl()
    control.connect()
    control.open(OUTPUT_TNT_PATH)
    tnmr_saved_tnt_path = OUTPUT_DIR / "20260620_NVNMR_template_waveform_edit_tnmr_saved.tnt"
    control.save_as(tnmr_saved_tnt_path)
    print("TNMR saved TNT:", tnmr_saved_tnt_path)
else:
    print("TNMR COM save skipped. Set RUN_TNMR_COM_SAVE = True to run it.")


## 15. Output Checklist

Use this final cell as the management-facing handoff summary: which files were generated, which parameters changed, which table was replaced, and what should be checked in TNMR.


In [ ]:
outputs_df = pd.DataFrame(
    [
        {"file": OUTPUT_TPS_PATH.name, "path": str(OUTPUT_TPS_PATH), "bytes": OUTPUT_TPS_PATH.stat().st_size},
        {"file": OUTPUT_TNT_PATH.name, "path": str(OUTPUT_TNT_PATH), "bytes": OUTPUT_TNT_PATH.stat().st_size},
        {"file": OUTPUT_JSON_PATH.name, "path": str(OUTPUT_JSON_PATH), "bytes": OUTPUT_JSON_PATH.stat().st_size},
    ]
)

display(outputs_df)
display(parameter_diff_df)
print("Waveform table:", TARGET_TABLE_NAME)
print("Waveform points:", waveform.summary().point_count)
print("TNMR acceptance check: open the generated TPS/TNT, inspect the sequence table, and confirm the phase-modulation table values.")
